# 按 QID 预览湖内图片数据

给定 Wikidata qid,从**合并主表 `images.v2`** 找到该概念的已收图片行,再从 **SG COS** 桶拉 blob 出图墙。

- **主账本 = `kb/images.v2.jsonl.gz`**(2026-09-23 并账发布,唯一权威):1,601.7 万行,**一图一行**、`qids`/`refs` 嵌套;毒页已全部分流到 `kb/quarantine/deadletter.v2.jsonl.gz`,毒 blob 已清理——**v2 里的行 blob 都在库,不会再缺图**。
- 可选对照源(`ledgers=` 参数):`wm` = 旧 v1 主账本(886 万行,~89% 毒行,仅对照);`df20`/`plantnet`/`pubchem`/`sdc_attach` = `kb/qid_images_ext/` 旧边表。
- 图片: `datasets/demiwtg/kb/<blob_path>`,`blobs/` 404 时自动试 `blobs-nc/`(tmdb 分区)
- 卡片里是**下载原图后本地压的缩略图**(默认最长边 360px/JPEG70,控制笔记本体积,不是外链预览);
  角标的"原图 W×H"是**缩略前实测的真实下载分辨率**(非显示尺寸),体积为 blob 实际字节数;
  点卡片 = 打开 **COS 原图签名直链**(15 分钟有效,浏览器看全分辨率);`max_edge=720/1024` 可调缩略清晰度。注意 sdc 线存的本身就是 thumb1920 档。。

**用法**:跑完上面三个 cell 后,在最后一个 cell 改 qid / 张数(`show_qid("Q1000404", n=8)`)即可;`peek_records("Q1000404")` 打原始账本记录。

- v2 首次使用需拉 2.0GB 账本到本地缓存(约几分钟),之后按 qid 扫描 ~1–2 分钟/批,结果落 `/tmp/qid_preview_cache/`,二次查询秒回。
> kernel 需带 Pillow + IPython(如 `Python 3 (demiwtg env)`);凭证走 `COS_SECRET_ID/KEY` 环境变量或 `sid:skey` 凭证文件(`/tmp/cos_creds`)。

In [ ]:
# ---- 配置 + 自包含 COS 只读客户端(签名拼法同 demiflow.collect.cosio, 勿改三处一致性) ----
import os, re, sys, json, gzip, html, time, base64, hashlib, hmac
import io as _io
import urllib.request, urllib.parse, urllib.error
from pathlib import Path

SRC_HOST = "lhcos-368f6-1256345599.cos.ap-singapore.myqcloud.com"   # sg 桶
SRC_ROOT = "lhcos-data/demiwtg-data"
KB = f"{SRC_ROOT}/datasets/demiwtg/kb"
CACHE = Path("/tmp/qid_preview_cache")              # 按 qid 的行缓存 + 账本原文缓存

def discover_creds():
    sid, skey = os.environ.get("COS_SECRET_ID", ""), os.environ.get("COS_SECRET_KEY", "")
    if sid and skey:
        return sid, skey
    for p in (os.path.expanduser("~/.config/demiflow/cos_creds"), "/tmp/cos_creds"):
        if os.path.exists(p):
            sid, _, skey = open(p).read().strip().partition(":")
            if sid and skey:
                return sid, skey
    raise RuntimeError("无 COS 凭证: 需 COS_SECRET_ID/KEY 环境变量, 或 sid:skey 凭证文件(/tmp/cos_creds)")

_SID, _SKEY = discover_creds()

def _sign(method, path, params, host=SRC_HOST):
    """COS 请求签名(q-url-param-list 必须与 FormatString 参数集一致——只此一种拼法;
    host 参与签名串, 请求哪个桶就签哪个 host)。"""
    now = int(time.time())
    kt = f"{now - 60};{now + 900}"
    sk = hmac.new(_SKEY.encode(), kt.encode(), hashlib.sha1).hexdigest()
    p = "&".join(f"{k.lower()}={urllib.parse.quote(str(v), safe='')}"
                 for k, v in sorted(params.items()))
    hs = f"{method.lower()}\n{path}\n{p}\nhost={host}\n"
    sts = f"sha1\n{kt}\n{hashlib.sha1(hs.encode()).hexdigest()}\n"
    sigv = hmac.new(sk.encode(), sts.encode(), hashlib.sha1).hexdigest()
    declared = ";".join(sorted(k.lower() for k in params))
    return (f"q-sign-algorithm=sha1&q-ak={_SID}&q-sign-time={kt}&q-key-time={kt}"
            f"&q-header-list=host&q-url-param-list={declared}&q-signature={sigv}")

def cos_get(key, timeout=180, retries=3):
    """GET 对象: 200→bytes, 404→None, 瞬态(403/429/5xx/传输错)退避重试后上抛。"""
    path = urllib.parse.quote("/" + key.lstrip("/"))
    url = f"https://{SRC_HOST}{path}"
    last = None
    for i in range(retries):
        try:
            req = urllib.request.Request(url, headers={"authorization": _sign("GET", path, {})})
            with urllib.request.urlopen(req, timeout=timeout) as r:
                return r.read()
        except urllib.error.HTTPError as e:
            if e.code == 404:
                return None
            last = e
        except Exception as e:
            last = e
        time.sleep(3 * (i + 1))
    raise last

print("COS 就绪:", SRC_HOST)

In [ ]:
# ---- 账本: qid → 行(默认合并主表 images.v2; 旧 v1/ext 边表仅对照) ----
# 账本注册表: 名字 → (COS key, 行形态)
#   v2  : images.v2.jsonl.gz   一图一行, qid 在 qids[] 数组里(2026-09-23 并账, 唯一权威)
#   wm  : qid_images.jsonl.gz  旧 v1 主账本, 一行一 (qid,图), ~89% 毒行(对照用)
#   ext : qid_images_ext/<t>.jsonl.gz  一行一 (qid,图)
LEDGERS = {
    "v2":  {"key": f"{KB}/images.v2.jsonl.gz", "kind": "qids_array"},
    "wm":  {"key": f"{KB}/qid_images.jsonl.gz", "kind": "per_qid"},
    "df20": {"key": f"{KB}/qid_images_ext/df20.jsonl.gz", "kind": "per_qid"},
    "plantnet": {"key": f"{KB}/qid_images_ext/plantnet.jsonl.gz", "kind": "per_qid"},
    "pubchem": {"key": f"{KB}/qid_images_ext/pubchem.jsonl.gz", "kind": "per_qid"},
    "sdc_attach": {"key": f"{KB}/qid_images_ext/sdc_attach.jsonl.gz", "kind": "per_qid"},
}
DEFAULT_LEDGERS = ["v2"]
LOCAL_FALLBACK = {"wm": "/tmp/qid_images.jsonl.gz"}   # 湖侧旧副本(有则免拉 COS)
# 国内镜像(sgx 中转 demiwtg-adhoc/): 大账本跨境直拉劣化时从 GZ 桶走, 秒级;
# 镜像不在则自动回退 sg 直连。需要新镜像时: 在 sgx 上把 sg COS 文件 PUT 到
# lhcos-cee54 桶 demiwtg-adhoc/ 前缀, 并在下面注册。
GZ_HOST = "lhcos-cee54-1256345599.cos.ap-guangzhou.myqcloud.com"
GZ_MIRROR = {"v2": "demiwtg-adhoc/images.v2.jsonl.gz"}

import threading, queue as _queue

def _par_download(key, out, host=SRC_HOST, chunk=64 << 20, n_threads=8):
    """Range 分段并行下载(跨境劣化期单流可能 <20KB/s, 并行分段才拉得动大账本);
    .part 落盘, 全部段到齐才转正——半截文件绝不被当完整缓存。"""
    path = urllib.parse.quote("/" + key.lstrip("/"))
    req = urllib.request.Request(f"https://{host}{path}",
                                 headers={"authorization": _sign("HEAD", path, {}, host)}, method="HEAD")
    with urllib.request.urlopen(req, timeout=60) as r:
        total = int(r.headers["Content-Length"])
    ranges = [(a, min(a + chunk, total) - 1) for a in range(0, total, chunk)]
    tasks = _queue.Queue()
    for i, rg in enumerate(ranges):
        tasks.put((i, rg))
    parts, lock, n_ok = [None] * len(ranges), threading.Lock(), [0]
    t0 = time.time()

    def _worker():
        path = urllib.parse.quote("/" + key.lstrip("/"))
        url = f"https://{host}{path}"
        while True:
            try:
                i, (a, b) = tasks.get_nowait()
            except _queue.Empty:
                return
            for attempt in range(5):
                try:
                    r = urllib.request.Request(url, headers={
                        "authorization": _sign("GET", path, {}, host),
                        "Range": f"bytes={a}-{b}"})
                    with urllib.request.urlopen(r, timeout=300) as resp:
                        data = resp.read()
                    assert len(data) == b - a + 1
                    parts[i] = data
                    with lock:
                        n_ok[0] += len(data)
                        el = max(time.time() - t0, 1e-6)
                        print(f"\r  [{key.split('/')[-1]}] {n_ok[0]/1e6:.0f}/{total/1e6:.0f}MB"
                              f" {n_ok[0]/1e6/el:.1f}MB/s", end="", flush=True)
                    break
                except Exception:
                    time.sleep(2 * (attempt + 1))
            else:
                raise RuntimeError(f"段{i} 重试耗尽")
    ths = [threading.Thread(target=_worker, daemon=True) for _ in range(n_threads)]
    for t in ths:
        t.start()
    for t in ths:
        t.join()
    if any(p is None for p in parts):
        raise RuntimeError("分段下载不完整(有线程重试耗尽), 重跑本 cell 即续")
    out.parent.mkdir(parents=True, exist_ok=True)
    with open(out.with_suffix(out.suffix + ".part"), "wb") as f:
        for p in parts:
            f.write(p)
    assert out.with_suffix(out.suffix + ".part").stat().st_size == total
    out.with_suffix(out.suffix + ".part").rename(out)
    print(f"\n  账本落盘 {out.name} {total/1e6:.0f}MB")

def _head(host, key):
    path = urllib.parse.quote("/" + key.lstrip("/"))
    req = urllib.request.Request(f"https://{host}{path}",
                                 headers={"authorization": _sign("HEAD", path, {}, host)},
                                 method="HEAD")
    try:
        with urllib.request.urlopen(req, timeout=30) as r:
            return int(r.headers.get("Content-Length") or 0)
    except Exception:
        return 0

def _table_file(ledger):
    """账本 gz 本地化: 本地副本 → GZ 镜像(在则走国内快道) → sg 直连;结果进 CACHE/_tables/。"""
    spec = LEDGERS[ledger]
    local = CACHE / "_tables" / f"{ledger}.jsonl.gz"
    if local.exists():
        return local
    fb = LOCAL_FALLBACK.get(ledger)
    if fb and os.path.exists(fb):
        return Path(fb)
    host, key = SRC_HOST, spec["key"]
    gz = GZ_MIRROR.get(ledger)
    if gz and _head(GZ_HOST, gz):
        host, key = GZ_HOST, gz
        print(f"[{ledger}] 走 GZ 镜像(国内快道)")
    print(f"[{ledger}] 拉账本 {key.split('/')[-1]} …")
    _par_download(key, local, host=host)
    return local

def _scan_gz(path, qids, kind):
    """流式扫 gz jsonl;qid 全词 substring 预筛, 命中才 json 解析校验。"""
    pats = {q: (f'"qid": "{q}"' if kind == "per_qid" else f'"{q}"') for q in qids}
    want = set(qids)
    out = {q: [] for q in qids}
    t0 = time.time()
    with gzip.open(path, "rt", encoding="utf-8", errors="replace") as f:
        for line in f:
            for q, pat in pats.items():
                if pat in line:
                    try:
                        rec = json.loads(line)
                    except json.JSONDecodeError:
                        break
                    hit = (rec.get("qid") == q) if kind == "per_qid" \
                        else (q in (rec.get("qids") or []))
                    if hit:
                        out[q].append(rec)
                    break
    print(f"    扫描完成 {time.time() - t0:.0f}s")
    return out

def _normalize(rec, ledger, qid):
    """各账本原始行 → 统一 {qid, source, blob_path, meta}(meta 保留原始字段)。"""
    if ledger == "v2":
        src = rec.get("source") or "wm"
        blob = rec.get("blob_path") or ""
        meta = {k: v for k, v in rec.items() if k not in ("blob_path", "qids")}
        meta["qids_n"] = len(rec.get("qids") or [])
        meta["qids_head"] = (rec.get("qids") or [])[:8]
    elif ledger == "wm":
        src, blob = "wm", rec.get("path") or ""
        meta = {k: v for k, v in rec.items() if k not in ("qid", "path")}
    else:
        src, blob = rec.get("source") or ledger, rec.get("blob_path") or rec.get("path") or ""
        meta = {k: v for k, v in rec.items() if k not in ("qid", "path", "blob_path")}
    if not blob.startswith("blobs"):
        blob = f"blobs/{rec['sha256'][:2]}/{rec['sha256']}.{rec.get('ext') or 'jpg'}"
    return {"qid": qid, "ledger": ledger, "source": src, "blob_path": blob, "meta": meta}

def _cache_path(ledger, qid):
    return CACHE / ledger / f"{qid}.json"

def _save_and_merge(ledger, got, rows):
    for q, recs in got.items():
        p = _cache_path(ledger, q)
        p.parent.mkdir(parents=True, exist_ok=True)
        uni = [_normalize(r, ledger, q) for r in recs]
        p.write_text("\n".join(json.dumps(u, ensure_ascii=False) for u in uni))
        rows.setdefault(q, []).extend(uni)
        print(f"  [{ledger}] {q}: +{len(uni)} 行")

def load_rows(qids, ledgers=None, refresh=False):
    """{qid: [统一行...]};已缓存的 qid 不再扫(refresh=True 强制重扫)。"""
    ledgers = ledgers or DEFAULT_LEDGERS
    qids = list(dict.fromkeys(qids))
    rows = {q: [] for q in qids}
    for lg in ledgers:
        todo = [q for q in qids if refresh or not _cache_path(lg, q).exists()]
        for q in qids:
            if q not in todo:
                rows[q] += [json.loads(l) for l in _cache_path(lg, q).read_text().splitlines() if l]
        if not todo:
            continue
        try:
            path = _table_file(lg)
        except Exception as e:
            print(f"[{lg}] 账本不可得({e}), 跳过")
            continue
        print(f"[{lg}] 扫 {path}({len(todo)} 个 qid)…")
        _save_and_merge(lg, _scan_gz(path, todo, LEDGERS[lg]["kind"]), rows)
    return rows

def peek_records(qid, ledgers=None, max_show=8, measure=True):
    """打原始账本记录(json)。
    v2 的 width/height 本身多为 null(并账原料 manifest 不带宽高, MERGE_SPEC 只 coalesce
    非 null 值)——measure=True 时下载 blob 解码, 在记录头部补"wh实测"真实分辨率。"""
    for lg in (ledgers or DEFAULT_LEDGERS):
        rows = load_rows([qid], ledgers=[lg])[qid]
        print(f"== [{lg}] {qid}: {len(rows)} 行 ==")
        for r in rows[:max_show]:
            m = dict(r["meta"])
            head = {"_source": r["source"]}
            if measure:
                data = _blob_bytes(r)
                if data is None:
                    head["wh实测"] = "blob 缺失"
                else:
                    try:
                        with Image.open(_io.BytesIO(data)) as im:
                            head["wh实测"] = f"{im.size[0]}×{im.size[1]}"
                    except Exception:
                        head["wh实测"] = "解码失败"
            print(json.dumps({**head, "blob_path": r["blob_path"], **m},
                             ensure_ascii=False)[:700])
        if len(rows) > max_show:
            print(f"… 还有 {len(rows) - max_show} 行")

print("账本加载器就绪: 默认", DEFAULT_LEDGERS, "| 可选", list(LEDGERS))

In [ ]:
# ---- 展示: 拉 blob → 缩略图墙(解码失败标红 = 坏图; 404 = COS 无此对象) ----
from IPython.display import HTML, display
from PIL import Image

_CSS = """<style>
.qgrid{display:flex;flex-wrap:wrap;gap:10px;font:12px/1.45 -apple-system,'PingFang SC',sans-serif}
.qcard{background:#fff;border:1px solid #e2e2e2;border-radius:8px;overflow:hidden;width:230px}
.qcard img{width:230px;height:172px;object-fit:cover;display:block;background:#f2f2f2}
.qmeta{padding:6px 8px;color:#444;word-break:break-all}
.src{background:#5b8def;color:#fff;border-radius:8px;font-size:10px;padding:1px 7px;margin-right:5px}
.led{background:#8a8a8a;color:#fff;border-radius:8px;font-size:10px;padding:1px 7px;margin-right:5px}
.qbad{width:230px;height:172px;display:flex;align-items:center;justify-content:center;
background:#fdecea;color:#b71c1c;font-size:11px}
h3.q{margin:20px 0 8px;font:600 15px/1.4 -apple-system,sans-serif}
.qsum{color:#777;font-size:12px;margin:2px 0 10px}
</style>"""

def _thumb_uri(blob, max_edge=360, quality=70):
    im = Image.open(_io.BytesIO(blob))
    im.load()
    orig_wh = im.size               # 真实原图分辨率(缩略前实测, 角标展示用)
    im = im.convert("RGB")
    im.thumbnail((max_edge, max_edge))
    buf = _io.BytesIO()
    im.save(buf, "JPEG", quality=quality)
    return f"data:image/jpeg;base64,{base64.b64encode(buf.getvalue()).decode()}", orig_wh

def _signed_url(row):
    """COS 原图签名直链(15 分钟内点开有效, 浏览器直看全分辨率)。"""
    key = f"{KB}/{row['blob_path']}"
    path = urllib.parse.quote("/" + key.lstrip("/"))
    return f"https://{SRC_HOST}{path}?{_sign('GET', path, {})}"

def _card(row, status, uri, real_wh, link=""):
    m = row["meta"]
    name = m.get("commons_file") or m.get("external_id") or m.get("orig_path")
    if not name and isinstance(m.get("refs"), list) and m["refs"]:
        name = m["refs"][0].get("external_id")
    name = name or ""
    dims = f"{m.get('width')}×{m.get('height')}" if m.get("width") else "—"
    size = m.get("size_bytes") or m.get("page_bytes") or 0
    url = m.get("content_url") or ((m.get("refs") or [{}])[0].get("orig_url", "") if isinstance(m.get("refs"), list) else "")
    if status == "ok":
        href = link or url
        img = (f'<a href="{html.escape(href)}" target="_blank"><img src="{uri}"></a>'
               if href else f'<img src="{uri}">')
        foot = f"原图 {real_wh[0]}×{real_wh[1]} · {int(size) // 1024}KB"
    else:
        img = '<div class="qbad">' + ("坏图<br>解码失败" if status == "poison" else "COS 无此对象") + "</div>"
        foot = "—" if status == "poison" else f"{int(size) // 1024}KB"
    sha8 = row["blob_path"].rpartition("/")[2][:8]
    led = (f'<span class="led">{row["ledger"]}</span>' if row["ledger"] != "v2" else "")
    return (f'<div class="qcard">{img}<div class="qmeta">{led}'
            f'<span class="src">{row["source"]}</span><b>{html.escape(str(name)[:44])}</b><br>'
            f"{foot} · 账本 {dims} · <code>{sha8}</code></div></div>")

def _blob_bytes(row):
    """kb/<blob_path>;blobs/ 404 → 试 blobs-nc/(b4 分区同 sha 布局)。"""
    data = cos_get(f"{KB}/{row['blob_path']}")
    if data is None and row["blob_path"].startswith("blobs/"):
        data = cos_get(f"{KB}/{row['blob_path'].replace('blobs/', 'blobs-nc/', 1)}")
    return data

def show_qid(qid, n=8, ledgers=None, refresh=False, max_edge=360, quality=70):
    """单个 qid: 出最多 n 张图;卡片=缩略图(点开=COS 原图签名直链),
    max_edge/quality 调缩略清晰度(如 max_edge=1024)。"""
    rows = load_rows([qid], ledgers=ledgers, refresh=refresh)[qid]
    if not rows:
        display(HTML(f"{_CSS}<h3 class='q'>{qid}</h3>"
                     f"<p class='qsum'>所选账本无此 qid</p>"))
        return
    by_src = {}
    for r in rows:
        by_src[r["source"]] = by_src.get(r["source"], 0) + 1
    stat = {"ok": 0, "poison": 0, "missing": 0}
    cards = []
    for row in rows[:n]:
        data = _blob_bytes(row)
        if data is None:
            stat["missing"] += 1
            cards.append(_card(row, "missing", None, None))
            continue
        try:
            uri, wh = _thumb_uri(data, max_edge=max_edge, quality=quality)
        except Exception:
            stat["poison"] += 1
            cards.append(_card(row, "poison", None, None))
            continue
        stat["ok"] += 1
        cards.append(_card(row, "ok", uri, wh, link=_signed_url(row)))
    more = len(rows) - min(n, len(rows))
    src_txt = " · ".join(f"{s}({c})" for s, c in by_src.items())
    extra = f",还有 {more} 行未展示" if more > 0 else ""
    tail = f" | 本页: ✅{stat['ok']} 毒{stat['poison']} 缺{stat['missing']}{extra}"
    display(HTML(f"{_CSS}<h3 class='q'>{qid} <small style='color:#888'>共 {len(rows)} 行</small></h3>"
                 f"<p class='qsum'>{src_txt}{tail}</p>"
                 f'<div class="qgrid">{"".join(cards)}</div>'))

def show_qids(qids, n=8, ledgers=None, refresh=False, max_edge=360, quality=70):
    """批量: 每个 qid 最多 n 张。"""
    for q in qids:
        show_qid(q, n=n, ledgers=ledgers, refresh=refresh,
                 max_edge=max_edge, quality=quality)

print("展示函数就绪: show_qid(…, max_edge=1024 可调清晰度) / show_qids / peek_records")

## 预览

改下面的 qid 与张数直接跑。默认账本 = 合并主表 v2(不缺图);想对照旧账本加 `ledgers=["v2","wm"]`。

In [ ]:
# ↓↓↓ 改这里 ↓↓↓
peek_records("Q1000404")                # 先看原始账本记录
show_qid("Q1000404", n=8)               # 再出图墙(最多 n 张)

# 批量 + 指定张数/账本:
# show_qids(["Q1000404", "Q12954047", "Q5122056", "Q161580"], n=4)
# show_qid("Q1000404", n=8, ledgers=["v2", "wm"])   # 对照旧 v1(毒行标红属预期)
# show_qid("Q1000404", n=8, refresh=True)           # 忽略缓存重扫账本
# show_qid("Q1000404", n=4, max_edge=1024)     # 高清缩略(卡片更清晰, 点开仍是原图直链)

## 备注

- **images.v2**(唯一权威主表):1,601.7 万行一图一行,`qids`/`refs` 嵌套;毒页 534,392 行已分流 `kb/quarantine/deadletter.v2.jsonl.gz`,毒 blob 与旧 rendition 共 1,163 万个已于 09-23 清理,库终态 1,759 万对象全部有账——所以 **v2 行的 blob 都在库**。旧 v1 主账本(886 万行,~89% 毒行)与 ext 边表仅作对照,缺图/标红属预期。
- **缓存**:`/tmp/qid_preview_cache/<ledger>/<qid>.json`(行缓存)与 `_tables/`(账本原文,v2 约 2GB)。清空即重扫:`rm -rf /tmp/qid_preview_cache`。
- **v2 的 width/height 为 null**:并账原料(各线 worker manifest)多不带宽高,MERGE_SPEC 口径只 coalesce 非 null 值,所以元数据看不出分辨率——以卡片角标 / `peek_records(measure=True)` 的解码实测为准(需要的话可按此回填 v2)。
- **只读**:本 notebook 对 COS 只做签名 GET,不写不删。